## MODELO LINEAR MISTO

O modelo linear misto (LMM) é um modelo linear, frequentemente usado para trabalhar com dados de medidas repetidas ou longitudinais. Ele é uma extenção do modelo liner geral que possibilita trabalhar tanto com parametros populacionais (efeitso fixos) quanto com individuais (efeitos aleatórios), além do erro experimental. 

A equação fundamental de um **Modelo Linear Misto** pode ser definido como:

$$Y_i=X_iβ+Z_y+e_i$$

Onde:

β representa todos os termos fixos

y representa os termos aleatórios

e representa o erro/resíduo

### Para entender melhor o modelo linear misto, elaboramos o material didático a seguir para esclarecer eventuais dúvidas práticas.


### **Guia Didático sobre Modelos Lineares Mistos (MLM):**

#### **Introdução:**

### **Tópico 1: Efeitos Fixos e Efeitos Aleatórios**

A principal distinção do MLM é sua capacidade de separar a variância dos dados em duas fontes: efeitos fixos e efeitos aleatórios.

**Efeitos Fixos (Fixed Effects):**

Os efeitos fixos são as variáveis que você considera como "preditoras". São os fatores cujo impacto médio na população você deseja quantificar. Eles representam perguntas como:

  * "Qual é o efeito de um aumento de 1°C na temperatura sobre a distância percorrida em alta intensidade?"
  * "Os jogadores correm menos no segundo tempo em comparação com o primeiro?"
  * "A umidade relativa do ar influencia a velocidade máxima atingida?"

No nosso script R, variáveis como `Período`, `Temp` (Temperatura), `UR` (Umidade Relativa) e `TG` (Temperatura de Globo) são tratadas como efeitos fixos e estamos interessados em estimar um coeficiente único e "fixo" para cada uma delas, que represente seu efeito médio em todos os jogadores.

**Efeitos Aleatórios: A Origem da Variabilidade Não Explicada**

Os efeitos aleatórios não são sobre estimar um impacto médio, mas sim sobre **modelar a estrutura de variabilidade** nos seus dados que não é capturada pelos efeitos fixos. Eles reconhecem que seus dados estão agrupados. Em vez de estimar um valor para cada jogador, o modelo estima a **variância** entre eles.

  * **Exemplo Prático (Fisiologia):** Imagine que você está testando o efeito de um suplemento de creatina no desempenho em sprints repetidos. O suplemento (grupo tratamento vs. placebo) é o **efeito fixo**. No entanto, você sabe que os atletas não são iguais. O Atleta A pode ter uma capacidade anaeróbica naturalmente maior que o Atleta B. Se você medir o desempenho deles em 10 sprints, as 10 medidas do Atleta A estarão, em média, correlacionadas entre si e provavelmente serão superiores às do Atleta B, independentemente do suplemento. O `Atleta` é o seu **efeito aleatório**. Você não está interessado em quantificar a habilidade específica do "Atleta A", mas sim em controlar o fato de que as observações dele são agrupadas e contabilizar a variabilidade geral que existe *entre* os atletas.

Nos nossos dados `FCWC25-dados-fisicos.csv`, os candidatos perfeitos para efeitos aleatórios são `Player_id` e/ou `Team_ide`. As várias linhas de dados de um mesmo `Player_id` não são independentes.

O código em R para isso, usando a biblioteca `lme4`, teria a seguinte sintaxe:

```r
# Modelo hipotético
# VD ~ EfeitoFixo1 + EfeitoFixo2 + (1 | EfeitoAleatorio)
# Onde (1 | ...) especifica um intercepto aleatório

# Exemplo com seus dados:
# Prever a distância total baseando-se na temperatura,
# controlando para a variabilidade individual de cada jogador.
modelo <- lmer(Total_Distance_m ~ Temp + (1 | Player_id), data = df)
```

O termo `(1 | Player_id)` instrui o modelo: "Assuma que cada jogador tem seu próprio intercepto (linha de base) de performance. Estime a variância desses interceptos."

-----

### **Tópico 2: O Intercepto**

Em qualquer modelo linear, o intercepto é o valor previsto para a variável dependente quando todos os preditores (efeitos fixos) são iguais a zero.

No contexto de dados de exercício, uma temperatura de 0°C ou um tempo de jogo de 0 minutos raramente faz sentido prático. Por isso, uma prática comum e extremamente útil, que você é aplicado no nosso script com as variáveis `_c`, é a **centralização na média**. Ao subtrair a média de uma variável preditora (como `Temp_c = Temp - mean(Temp)`), o novo "zero" dela passa a ser a média original.

**Interpretação Prática:** Quando você usa variáveis centralizadas, o **intercepto do modelo representa o valor esperado da variável dependente (ex: `distancia_alta_intensidade`) para um jogador médio, sob as condições médias dos seus preditores** (ex: na temperatura média, umidade média, etc.) e no nível de referência de suas variáveis categóricas (ex: `Período` = "1º Tempo").

Isso transforma o intercepto de um valor abstrato para um ponto de referência extremamente útil e interpretável no contexto do seu estudo.

-----

### **Tópico 3: Métricas de Qualidade do Modelo - AIC, BIC e R²**

Quando voce estiver fazendo a sua análise, voce deve se perguntar: O modelo foi criado. Mas ele é bom? E se eu tivermos vários modelos, qual escolher?

**AIC (Akaike Information Criterion) e BIC (Bayesian Information Criterion)**

Pense no AIC e no BIC como medidas de "custo-benefício" de um modelo. Eles buscam um equilíbrio entre a **qualidade do ajuste** do modelo aos dados e sua **complexidade** (número de parâmetros estimados). Um modelo com mais variáveis quase sempre se ajustará melhor aos dados, mas corre o risco de "overfitting" (capturar ruído aleatório em vez do sinal verdadeiro).

  * **O que significam:** Ambos penalizam o modelo por adicionar mais parâmetros. O BIC aplica uma penalidade mais rigorosa por complexidade do que o AIC.
  * **Como usar:** **Eles não têm um significado absoluto.** Seu poder está na **comparação relativa entre modelos aninhados** (modelos que são subconjuntos um do outro). Ao comparar o `Modelo A` com o `Modelo B`, aquele com o **menor valor de AIC ou BIC é considerado o mais parcimonioso**, ou seja, o que melhor explica os dados com o mínimo de complexidade.

**Exemplo de uso:**

1.  `Modelo Nulo <- lmer(Distancia_alta_intensidade ~ 1 + (1 | Player_id), data = df)`
2.  `Modelo Temp <- lmer(Distancia_alta_intensidade ~ Temp_c + (1 | Player_id), data = df)`
3.  `Modelo Completo <- lmer(Distancia_alta_intensidade ~ Temp_c + UR_c + (1 | Player_id), data = df)`

Você compararia o AIC de cada um. Se o AIC do `Modelo Temp` for substancialmente menor que o do `Modelo Nulo`, isso sugere que adicionar a temperatura melhora o modelo.

**R² Marginal e R² Condicional: Desvendando a Variância**

Em MLM, o R² tradicional é dividido em dois para nos dar uma visão muito mais rica.

  * **R² Marginal ($R^2\_m$):** Representa a proporção da variância da variável dependente que é explicada **apenas pelos efeitos fixos**.

      * *Interpretação:* "Quanta da variação no desempenho físico dos jogadores pode ser atribuída a fatores como a temperatura, o período do jogo e a umidade do ar?"

  * **R² Condicional ($R^2\_c$):** Representa a proporção da variância explicada pelo modelo **inteiro**, ou seja, **pelos efeitos fixos e aleatórios juntos**.

      * *Interpretação:* "Quanta da variação no desempenho pode ser atribuída aos fatores ambientais E às diferenças individuais inerentes entre os jogadores?"

**A Diferença é a Chave:** A diferença entre $R^2\_c$ e $R^2\_m$ é incrivelmente informativa. Ela quantifica a proporção da variância que é devida à estrutura de agrupamento (as diferenças individuais entre jogadores/times). Se o seu $R^2\_m$ for 0.15 e o $R^2\_c$ for 0.70, isso indica que seus efeitos fixos (ambiente) explicam 15% da variância, mas a maior parte da variabilidade no desempenho (55%) é devida a diferenças consistentes entre os jogadores. Isso reforça a importância de usar um MLM em primeiro lugar\!

No R, o pacote `performance` faz esse cálculo facilmente:

```r
# Assumindo que 'mixed_model' é seu modelo lmer
r2(mixed_model)
```

**Tabela de Resultados (Exemplo de Interpretação)**

Suponha que seu modelo gere a seguinte saída para os efeitos fixos:

| Termo                  | Estimate | Std. Error | t.statistic | p.value |
| ---------------------- | -------- | ---------- | ----------- | ------- |
| (Intercept)            | 150.50   | 10.20      | 14.75       | \<.001   |
| Período (2º Tempo)     | -25.80   | 5.50       | -4.69       | \<.001   |
| Temp\_c (por 1°C)      | -3.10    | 1.50       | -2.07       | 0.040   |
| UR\_c (por 1%)         | -0.50    | 0.60       | -0.83       | 0.405   |

**Interpretação Didática:**

  * **Intercepto:** Em condições médias de temperatura e umidade, e durante o 1º tempo, um jogador percorre, em média, 150.50 metros em alta intensidade.
  * **Período:** Passar para o 2º tempo está associado a uma redução de 25.80 metros na distância percorrida em alta intensidade, em média, mantendo as outras variáveis constantes. Este efeito é estatisticamente significativo (p \< .001).
  * **Temperatura:** Para cada aumento de 1°C acima da média, a distância percorrida diminui em 3.10 metros, em média. Este é um efeito significativo (p = 0.040).
  * **Umidade Relativa:** A umidade relativa não parece ter um efeito estatisticamente significativo na distância percorrida neste modelo (p = 0.405).

-----

### **Tópico 4: Modelos com Interações**

Uma interação ocorre quando o efeito de uma variável preditora sobre a variável dependente **muda dependendo do nível de outra variável preditora**.

No nosso script R, nós analisamos a interação `Período ~ TG_c`. Isso testa a hipótese de que o efeito da temperatura de globo no desempenho não é o mesmo na noite e na tarde tempo. Por exemplo, talvez o efeito negativo da temperatura seja muito mais pronunciado na noite, quando os jogadores já estão fatigados.

**Interpretação:**
Se o termo de interação `Período2ºTempo:TG_c` for significativo, você não pode mais interpretar os efeitos principais isoladamente. A interpretação se torna:

  * "O efeito da temperatura no 1º tempo é X."
  * "O efeito da temperatura no 2º tempo é X + Y", onde Y é o coeficiente do termo de interação.

**Visualização é Essencial:** Interações são difíceis de interpretar apenas com números. Gráficos de efeitos, como os que nos gerou com `emmip`, são a melhor maneira de visualizar e compreender uma interação. Eles mostram as "slopes" (inclinações) de uma variável contínua (temperatura) para cada nível da variável categórica (período).


-----

### **Tópico 5: Finalizando com um Exercício Aplicado**

**Cenário do Estudo de Caso:**
Você está analisando dados de GPS de jogadores de meio-campo de elite. O objetivo é entender os preditores do número de "sprints" (corridas na Zona 5) por minuto de jogo. Você construiu o seguinte modelo linear misto:

`sprints_por_min ~ Posse_de_bola_c + Ranking_oponente_c + Posse_de_bola_c:Ranking_oponente_c + (1 | Player_id)`

  * **Variável Dependente:** `sprints_por_min`
  * **Efeitos Fixos:**
      * `Posse_de_bola_c`: Porcentagem de posse de bola do time, centralizada na média.
      * `Ranking_oponente_c`: Ranking FIFA do oponente, centralizado na média (um valor mais alto significa um oponente mais fraco).
      * A interação entre eles.
  * **Efeito Aleatório:** `Player_id` para controlar as diferenças individuais entre os meio-campistas.

**Tabela de Resultados do Modelo:**

| Termo                                  | Estimate | Std. Error | p.value |
| -------------------------------------- | -------- | ---------- | ------- |
| (Intercept)                            | 0.35     | 0.04       | \<.001   |
| Posse\_de\_bola\_c                       | -0.02    | 0.005      | \<.001   |
| Ranking\_oponente\_c                     | 0.008    | 0.003      | 0.007   |
| Posse\_de\_bola\_c:Ranking\_oponente\_c | -0.001   | 0.0002     | \<.001   |
| **Métricas do Modelo** |          |            |         |
| R² Marginal                            | 0.28     |            |         |
| R² Condicional                         | 0.65     |            |         |

**Perguntas para Interpretação:**

1.  **Intercepto:** O que o valor de 0.35 do intercepto representa neste contexto?
2.  **Efeitos Principais:** Antes de olhar a interação, qual parece ser o efeito de ter mais posse de bola e de jogar contra um oponente mais fraco (ranking mais alto)?
3.  **Interação:** O termo de interação é significativo. Como você começaria a interpretar o que `-0.001` significa na prática? Que tipo de gráfico você geraria para visualizar isso?
4.  **R²:** Explique o que o R² Marginal de 0.28 e o R² Condicional de 0.65 lhe dizem sobre os fatores que determinam o número de sprints. Qual é a importância prática da diferença entre eles?
5.  **Conclusão para o Treinador:** Com base nesses resultados, qual seria sua principal mensagem para a comissão técnica sobre o trabalho de sprint dos meio-campistas? (Dica: pense em como o estilo de jogo - posse de bola - muda o esforço físico dependendo da qualidade do adversário).